## Reduction Potential Predictor Test
Vivian Weigel - Spring 2025

#### Imports

In [1]:
import pandas as pd
from archived_files import config
from archived_files.config import CONFIG
from archived_files.data_processing import MolecularDataset
from archived_files.model import ReductionPotentialPredictor
from archived_files.model_training import ModelTrainer
import torch

#### Data Processing

In [2]:
df = pd.read_json("../data/processed_dataset.json")
df.head()

,inchi_key,smiles,homo (eV),lumo (eV),vertical_excitation_energy (eV),zpve_S0 (eV),zpve_S1 (eV),zpve_T1 (eV),total_electronic_energy_S0 (eV),total_electronic_energy_S1 (eV),...,oxidation_potential_S1 (eV),oxidation_potential_T1 (eV),optimized_coordinates_S0,optimized_coordinates_S1,optimized_coordinates_T1,free_energy_S0,free_energy_AN,free_energy_CAT,free_energy_S1,free_energy_T1
0,AAEBJCSCYHPBRF-FZSIALSZSA-N,O=C1C(=CC(=O)c2c1cccn2)/C=N/C(F)(F)F,-8.228179,-3.955447,2.6688,3.992917,3.907637,3.925379,-26699.17376,-26696.69680,...,-5.136716,-4.733036,"[23, O 0.413251 -2.27768 -0.00018, C 0.747671 ...","[23, O 0.413251 -2.27768 -0.00018, C 0.747671 ...","[23, O 0.413251 -2.27768 -0.00018, C 0.747671 ...",-981.071563,-981.241446,-980.794288,-980.986124,-981.001929
1,AAHGPKDQAXQSJI-UHFFFAOYSA-N,Clc1ccc([nH]1)C1=CC(=O)c2c(C1=O)cccn2,-6.378621,-3.549181,2.1410,4.731080,4.689855,4.684059,-33177.50831,-33175.70462,...,-2.823009,-2.078529,"[25, O 0.10543 -1.770961 0.000611, C -0.53255 ...","[25, O 0.10543 -1.770961 0.000611, C -0.53255 ...","[25, O 0.10543 -1.770961 0.000611, C -0.53255 ...",-1219.119925,-1219.269897,-1218.904554,-1219.054324,-1219.082456
2,AAHTVBYDUMLUIQ-UHFFFAOYSA-N,N#Cc1cnc2c(n1)C(=O)C(=C(C2=O)c1scc(n1)F)c1scc(...,-7.034688,-4.188104,2.0703,4.397768,4.362883,4.357006,-54229.95823,NaN,...,NaN,-2.553724,"[29, N 7.183749 0.980076 0.266238, C 6.096159 ...",None,"[29, N 7.183749 0.980076 0.266238, C 6.096159 ...",-1992.806134,-1992.978825,-1992.568290,-1992.744838,-1992.772121
3,AAIKASVITWZPIQ-UHFFFAOYSA-N,O=C1C=C(C(=O)c2c1nccc2)c1ccc(n1C)Cl,-6.412907,-3.495303,2.1421,5.477788,5.453298,5.436944,-34246.08266,NaN,...,NaN,-2.287236,"[28, C -2.129751 -1.3759 1.159809, N -2.370051...",None,"[28, C -2.129751 -1.3759 1.159809, N -2.370051...",-1258.363799,-1258.512421,-1258.146740,-1258.306310,-1258.320177
4,AAUFRJVSRZUMGK-UHFFFAOYSA-N,N#Cc1cc(C#N)nc2c1C(=O)C(=CC2=O)c1cnc([nH]1)N(=...,-7.695924,-4.191642,2.6871,4.670045,4.604901,4.595051,-31688.03885,-31685.68285,...,-4.664537,-3.887813,"[28, N 6.777509 -1.420033 0.03503, C 5.676869 ...","[28, N 6.777509 -1.420033 0.03503, C 5.676869 ...","[28, N 6.777509 -1.420033 0.03503, C 5.676869 ...",-1164.393533,-1164.571720,-1164.134304,-1164.315726,-1164.343203


In [3]:
# keep only used columns
columns = ["inchi_key", "homo (eV)", "lumo (eV)", "dipole_moment_vector_S1 (D)", "reduction_potential_S1 (eV)", "optimized_coordinates_S0"]
df = df[columns]
df = df.dropna()

# grab 10 rows for testing purposes
df = df.head(10)
df

# save to testing json
df.to_json("test_set_filtered.json", orient="records", indent=4)

In [4]:
# convert to dict for processing purposes
data_list = df.to_dict(orient='records')

# Create dataset instance
dataset = MolecularDataset(data_list)
print(dataset.print_sample(10))

# Initialize model
model = ReductionPotentialPredictor().to(torch.device(config.DEVICE))

Printing 10 sample molecules from the dataset:

Molecule 1:
 Reduction Potential (S1): 2.214430602
  Symbols: ['O', 'C', 'F', 'F', 'F', 'C', 'C', 'C', 'H', 'H']
  Positions:
    O: [0.413251, -2.27768, -0.00018]
    C: [-3.950389, 0.01734, -0.0]
    F: [-4.569819, 0.4961, -1.08063]
    F: [-4.136409, -1.31631, -0.00019]
    F: [-4.569459, 0.49566, 1.08109]
    C: [4.770821, 0.094699, 0.00011]
    C: [4.493591, -1.275741, 0.0001]
    C: [2.175001, -0.70142, -0.0]
    H: [-0.648399, 2.06752, -0.00013]
    H: [5.304461, -1.998481, 0.00012]

----------------------------------------

Molecule 2:
 Reduction Potential (S1): 1.043411653
  Symbols: ['O', 'C', 'C', 'C', 'C', 'C', 'C', 'Cl', 'N', 'C', 'C', 'O', 'C', 'N', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
  Positions:
    O: [0.10543, -1.770961, 0.000611]
    C: [-0.53255, -0.72396, 0.000211]
    C: [0.139031, 0.611839, 4.1]
    C: [1.580361, 0.687909, 3.1]
    C: [2.413251, 1.804968, 0.000191]
    C: [3.747201, 1.363458, 0.00

In [5]:
# Initialize and train
trainer = ModelTrainer(CONFIG)
trainer.run_training()

x[src] shape: torch.Size([3362, 5])
edge_attr shape: torch.Size([3362])


AssertionError: Incorrect last dimension for y